In [1]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from scipy.stats import ortho_group
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set paths
DATA_DIR = Path('./data')
OUTPUT_DIR = Path('./data')

print("Reversible Noise Transformation Pipeline Started")
print("=" * 50)

Reversible Noise Transformation Pipeline Started


## 1. Load Prepared Features

In [2]:
# Load features
with open(DATA_DIR / 'train_features.pkl', 'rb') as f:
    train_df = pickle.load(f)

with open(DATA_DIR / 'test_features.pkl', 'rb') as f:
    test_df = pickle.load(f)

print(f"Training data: {train_df.shape}")
print(f"Test data: {test_df.shape}")

# Extract numeric features only
feature_cols = [col for col in train_df.columns if col not in ['account', 'label']]
numeric_cols = train_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

print(f"\nNumeric features for transformation: {len(numeric_cols)}")

Training data: (17640, 1044)
Test data: (7558, 1044)

Numeric features for transformation: 1042


## 2. Standardize Features

Standardization is necessary before applying transformations to ensure:
- Features are on the same scale
- Rotations and scaling make sense mathematically
- Transformations are reversible

In [3]:
# Fit scaler on training data
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[numeric_cols])
test_scaled = scaler.transform(test_df[numeric_cols])

print(f"Scaled training features: {train_scaled.shape}")
print(f"Scaled test features: {test_scaled.shape}")
print(f"\nScaler mean: {scaler.mean_[:5]}...")
print(f"Scaler std: {scaler.scale_[:5]}...")

Scaled training features: (17640, 1042)
Scaled test features: (7558, 1042)

Scaler mean: [ 6660.29347725  6651.41204624 11159.19302721   223.26896423
   223.25904566]...
Scaler std: [128732.06371858 128700.4815045  325445.22855954   3714.51390464
   3714.50369378]...


## 3. Define Reversible Transformation Classes

In [4]:
class OrthogonalRotation:
    """
    Orthogonal rotation transformation.
    Preserves distances and angles - fully reversible.
    """
    def __init__(self, n_features, seed=42):
        np.random.seed(seed)
        # Generate random orthogonal matrix
        self.rotation_matrix = ortho_group.rvs(n_features)
        self.inverse_matrix = self.rotation_matrix.T  # For orthogonal matrices, inverse = transpose
    
    def transform(self, X):
        return X @ self.rotation_matrix
    
    def inverse_transform(self, X):
        return X @ self.inverse_matrix
    
    def verify_invertibility(self, X, tolerance=1e-10):
        """Verify transformation is truly reversible"""
        X_transformed = self.transform(X)
        X_recovered = self.inverse_transform(X_transformed)
        error = np.max(np.abs(X - X_recovered))
        return error < tolerance, error


class DiagonalScaling:
    """
    Diagonal scaling transformation.
    Scales each feature independently - reversible.
    """
    def __init__(self, n_features, scale_range=(0.5, 2.0), seed=42):
        np.random.seed(seed)
        # Random scaling factors for each feature
        self.scale_factors = np.random.uniform(scale_range[0], scale_range[1], n_features)
        self.inverse_factors = 1.0 / self.scale_factors
    
    def transform(self, X):
        return X * self.scale_factors
    
    def inverse_transform(self, X):
        return X * self.inverse_factors
    
    def verify_invertibility(self, X, tolerance=1e-10):
        X_transformed = self.transform(X)
        X_recovered = self.inverse_transform(X_transformed)
        error = np.max(np.abs(X - X_recovered))
        return error < tolerance, error


class AffineTransform:
    """
    Affine transformation: linear + translation.
    Fully reversible if linear part is invertible.
    """
    def __init__(self, n_features, seed=42):
        np.random.seed(seed)
        # Random invertible matrix (use small random perturbation of identity)
        self.linear = np.eye(n_features) + np.random.randn(n_features, n_features) * 0.1
        self.translation = np.random.randn(n_features) * 0.5
        
        # Compute inverse
        self.linear_inv = np.linalg.inv(self.linear)
    
    def transform(self, X):
        # X @ A^T + b
        return X @ self.linear.T + self.translation
    
    def inverse_transform(self, X):
        # (X - b) @ (A^-1)^T = (X - b) @ (A^T)^-1
        return (X - self.translation) @ self.linear_inv.T
    
    def verify_invertibility(self, X, tolerance=1e-8):
        X_transformed = self.transform(X)
        X_recovered = self.inverse_transform(X_transformed)
        error = np.max(np.abs(X - X_recovered))
        return error < tolerance, error


class CompositeTransform:
    """
    Composite of multiple transformations.
    Apply in sequence: T = T3(T2(T1(X)))
    Inverse: T^-1 = T1^-1(T2^-1(T3^-1(X)))
    """
    def __init__(self, transforms):
        self.transforms = transforms
    
    def transform(self, X):
        result = X.copy()
        for t in self.transforms:
            result = t.transform(result)
        return result
    
    def inverse_transform(self, X):
        result = X.copy()
        for t in reversed(self.transforms):
            result = t.inverse_transform(result)
        return result
    
    def verify_invertibility(self, X, tolerance=1e-7):
        X_transformed = self.transform(X)
        X_recovered = self.inverse_transform(X_transformed)
        error = np.max(np.abs(X - X_recovered))
        return error < tolerance, error

print("✓ Transformation classes defined")

✓ Transformation classes defined


## 4. Create Multiple Transformation Variants

Generate several transformation variants to create diverse noise-augmented features.

In [5]:
n_features = len(numeric_cols)
n_variants = 5  # Create 5 different noise variants

transforms = {}

for i in range(n_variants):
    seed = 42 + i
    
    # Variant 1: Pure rotation
    if i == 0:
        transform = OrthogonalRotation(n_features, seed=seed)
        name = 'rotation'
    
    # Variant 2: Pure scaling
    elif i == 1:
        transform = DiagonalScaling(n_features, scale_range=(0.7, 1.5), seed=seed)
        name = 'scaling'
    
    # Variant 3: Affine
    elif i == 2:
        transform = AffineTransform(n_features, seed=seed)
        name = 'affine'
    
    # Variant 4: Rotation + Scaling
    elif i == 3:
        transform = CompositeTransform([
            OrthogonalRotation(n_features, seed=seed),
            DiagonalScaling(n_features, scale_range=(0.8, 1.3), seed=seed)
        ])
        name = 'rotation_scaling'
    
    # Variant 5: Full composite
    else:
        transform = CompositeTransform([
            OrthogonalRotation(n_features, seed=seed),
            DiagonalScaling(n_features, scale_range=(0.9, 1.2), seed=seed),
            AffineTransform(n_features, seed=seed)
        ])
        name = 'composite'
    
    transforms[f'variant_{i}_{name}'] = transform
    print(f"✓ Created transformation variant {i}: {name}")

✓ Created transformation variant 0: rotation
✓ Created transformation variant 1: scaling

✓ Created transformation variant 1: scaling
✓ Created transformation variant 2: affine
✓ Created transformation variant 3: rotation_scaling
✓ Created transformation variant 2: affine
✓ Created transformation variant 3: rotation_scaling
✓ Created transformation variant 4: composite
✓ Created transformation variant 4: composite


## 5. Verify Invertibility

Critical step: Ensure all transformations are truly reversible.

In [6]:
# Test on sample data
test_sample = train_scaled[:100]  # Use first 100 samples

print("Verifying invertibility of all transformations...\n")
print(f"{'Variant':<25} {'Invertible':<15} {'Max Error':<15}")
print("-" * 55)

all_invertible = True
for name, transform in transforms.items():
    is_invertible, error = transform.verify_invertibility(test_sample)
    status = "✓ YES" if is_invertible else "✗ NO"
    print(f"{name:<25} {status:<15} {error:<15.2e}")
    all_invertible = all_invertible and is_invertible

print("-" * 55)
if all_invertible:
    print("\n✓ All transformations are invertible!")
else:
    print("\n✗ WARNING: Some transformations are not invertible!")

Verifying invertibility of all transformations...

Variant                   Invertible      Max Error      
-------------------------------------------------------
variant_0_rotation        ✓ YES           7.11e-15       
variant_1_scaling         ✓ YES           3.55e-15       
variant_2_affine          ✓ YES           1.01e-11       
variant_3_rotation_scaling ✓ YES           1.07e-14       
variant_4_composite       ✓ YES           4.98e-12       
-------------------------------------------------------

✓ All transformations are invertible!


## 6. Apply Transformations to Data

In [7]:
# Apply each transformation to both train and test
train_noisy = {}
test_noisy = {}

for variant_name, transform in transforms.items():
    print(f"Applying {variant_name}...")
    
    # Transform
    train_transformed = transform.transform(train_scaled)
    test_transformed = transform.transform(test_scaled)
    
    # Convert back to DataFrame with new column names
    train_noisy_df = pd.DataFrame(
        train_transformed,
        columns=[f'{col}_{variant_name}' for col in numeric_cols]
    )
    train_noisy_df['account'] = train_df['account'].values
    train_noisy_df['label'] = train_df['label'].values
    
    test_noisy_df = pd.DataFrame(
        test_transformed,
        columns=[f'{col}_{variant_name}' for col in numeric_cols]
    )
    test_noisy_df['account'] = test_df['account'].values
    
    train_noisy[variant_name] = train_noisy_df
    test_noisy[variant_name] = test_noisy_df
    
    print(f"  Train: {train_noisy_df.shape}, Test: {test_noisy_df.shape}")

print(f"\n✓ Created {len(train_noisy)} noise variants")

Applying variant_0_rotation...
  Train: (17640, 1044), Test: (7558, 1043)
Applying variant_1_scaling...
  Train: (17640, 1044), Test: (7558, 1043)
Applying variant_1_scaling...


  Train: (17640, 1044), Test: (7558, 1043)
Applying variant_2_affine...
  Train: (17640, 1044), Test: (7558, 1043)
Applying variant_3_rotation_scaling...
  Train: (17640, 1044), Test: (7558, 1043)
Applying variant_3_rotation_scaling...
  Train: (17640, 1044), Test: (7558, 1043)
Applying variant_4_composite...
  Train: (17640, 1044), Test: (7558, 1043)
Applying variant_4_composite...
  Train: (17640, 1044), Test: (7558, 1043)

✓ Created 5 noise variants
  Train: (17640, 1044), Test: (7558, 1043)

✓ Created 5 noise variants


## 7. Combine Original + Noisy Features

Create comprehensive feature sets that include both original and noise-augmented features.

In [8]:
# Start with original features
train_combined = train_df.copy()
test_combined = test_df.copy()

# Add each noise variant
for variant_name, train_variant in train_noisy.items():
    # Drop acc_id and label from variant (already in combined)
    variant_features = [col for col in train_variant.columns if col not in ['account', 'label']]
    train_combined = train_combined.merge(
        train_variant[['account'] + variant_features],
        on='account',
        how='left'
    )
    
    test_combined = test_combined.merge(
        test_noisy[variant_name][['account'] + variant_features],
        on='account',
        how='left'
    )

print(f"Combined training features: {train_combined.shape}")
print(f"Combined test features: {test_combined.shape}")
print(f"\nOriginal features: {len(numeric_cols)}")
print(f"Total features (with noise variants): {train_combined.shape[1] - 2}")  # -2 for acc_id and label

Combined training features: (17640, 6254)
Combined test features: (7558, 6254)

Original features: 1042
Total features (with noise variants): 6252


## 8. Save Outputs

In [9]:
# Save transformations for later analysis
transform_data = {
    'transforms': transforms,
    'scaler': scaler,
    'numeric_cols': numeric_cols,
    'n_variants': n_variants
}

with open(OUTPUT_DIR / 'noise_transforms.pkl', 'wb') as f:
    pickle.dump(transform_data, f)
print("✓ Saved noise_transforms.pkl")

# Save augmented features
with open(OUTPUT_DIR / 'train_features_noisy.pkl', 'wb') as f:
    pickle.dump(train_combined, f)
print(f"✓ Saved train_features_noisy.pkl ({train_combined.shape})")

with open(OUTPUT_DIR / 'test_features_noisy.pkl', 'wb') as f:
    pickle.dump(test_combined, f)
print(f"✓ Saved test_features_noisy.pkl ({test_combined.shape})")

print("\n" + "="*50)
print("Reversible Noise Transformation Complete!")
print("="*50)
print(f"\nCreated {n_variants} noise transformation variants")
print(f"Original features: {len(numeric_cols)}")
print(f"Augmented features: {train_combined.shape[1] - 2}")
print(f"Feature expansion: {(train_combined.shape[1] - 2) / len(numeric_cols):.1f}x")

✓ Saved noise_transforms.pkl
✓ Saved train_features_noisy.pkl ((17640, 6254))
✓ Saved train_features_noisy.pkl ((17640, 6254))
✓ Saved test_features_noisy.pkl ((7558, 6254))

Reversible Noise Transformation Complete!

Created 5 noise transformation variants
Original features: 1042
Augmented features: 6252
Feature expansion: 6.0x
✓ Saved test_features_noisy.pkl ((7558, 6254))

Reversible Noise Transformation Complete!

Created 5 noise transformation variants
Original features: 1042
Augmented features: 6252
Feature expansion: 6.0x
